# Align an LLM and Build RAG with Nugen API

In this tutorial, we'll walk through two key capabilities of the Nugen platform:

1. **Model Alignment** — Fine-tune a small LLM on a domain-specific document using the Nugen Alignment API
2. **RAG (Retrieval-Augmented Generation)** — Build a simple question-answering system that retrieves relevant context from the document and uses the aligned model to generate accurate answers

**What we'll use:**
- A summary of the **Indian Factories Act, 1948** as our domain document
- **Qwen 2.5 0.5B** as the base model for alignment (small and fast)
- **Nomic Embed Text v1.5** for generating text embeddings
- Pure Python with NumPy — no heavy dependencies like vector databases

By the end, you'll have a working pipeline that can answer questions about the Factories Act using your aligned model.

## Prerequisites

You'll need:
- A **Nugen API key** (get one free at [platform.nugen.in](https://platform.nugen.in))
- Python 3.8+ with `requests` and `numpy` installed

In [ ]:
# Install dependencies (skip if already installed)
!pip install requests numpy python-dotenv --quiet

## Step 1: Setup and Authentication

Load your Nugen API key. We recommend storing it in a `.env` file rather than hardcoding it.

Create a `.env` file in this directory:
```
NUGEN_API_KEY=your_api_key_here
```

In [ ]:
import os
import json
import time
import requests
import numpy as np
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("NUGEN_API_KEY")
BASE_URL = "https://api.nugen.in/api/v3"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# Quick check — make sure the key works
r = requests.get(f"{BASE_URL}/models/base", headers=HEADERS)
if r.status_code == 200:
    models = r.json()["models"]
    print(f"Connected to Nugen API. {len(models)} models available.")
else:
    print(f"Error: {r.status_code} — check your API key")

Let's see which models support alignment (fine-tuning):

In [ ]:
# Show alignment-ready models
for m in models:
    if m["alignment_ready"]:
        print(f"  {m['id']:40s} — {m['description']}")

We'll use **`qwen-v2p5-0p5b-instruct`** — a small 0.5B parameter model. It aligns quickly and is perfect for demonstrating the workflow.

---

## Step 2: Upload a Document

The Nugen alignment API learns from documents you upload. We'll use a summary of the **Indian Factories Act, 1948** — a real piece of Indian labor law covering health, safety, welfare, and working hours in factories.

Supported file types: `.txt`, `.json`, `.jsonl`, `.md`, `.xml`, `.csv`

In [ ]:
# Upload the document
doc_path = "indian_factories_act_summary.txt"

upload_headers = {"Authorization": f"Bearer {API_KEY}"}
files = [("files", (doc_path, open(doc_path, "rb"), "text/plain"))]

r = requests.post(f"{BASE_URL}/documents", headers=upload_headers, files=files)
upload_id = r.json()["document_ids"][0]
print(f"Upload ID: {upload_id}")

In [ ]:
# Wait for the document to finish processing
while True:
    r = requests.get(f"{BASE_URL}/documents/{upload_id}", headers=HEADERS)
    status = r.json()["status"]
    doc_id = r.json()["document_id"]
    print(f"Status: {status}")
    if status == "READY":
        break
    time.sleep(3)

print(f"\nDocument ready. Document ID: {doc_id}")

---

## Step 3: Align the Model

Now we create an alignment project. This tells Nugen to fine-tune `qwen-v2p5-0p5b-instruct` on our document so it becomes a specialist in the Factories Act.

In [ ]:
# Create alignment project
alignment_payload = {
    "name": "factories-act-alignment",
    "base_model": "qwen-v2p5-0p5b-instruct",
    "document_ids": [doc_id],
    "description": "Align Qwen 0.5B on Indian Factories Act for domain-specific QA",
}

r = requests.post(
    f"{BASE_URL}/alignment-project/create",
    headers=HEADERS,
    json=alignment_payload,
)

alignment_id = r.json()["alignment_id"]
print(f"Alignment started!")
print(f"Alignment ID: {alignment_id}")
print(f"Status: {r.json()['status']}")

In [ ]:
# Poll until alignment completes
# This typically takes 5-15 minutes for a small model + small document

print("Waiting for alignment to complete...")
print("(This may take 5-15 minutes)\n")

while True:
    r = requests.get(
        f"{BASE_URL}/alignment-project/status/{alignment_id}",
        headers=HEADERS,
    )
    result = r.json()
    status = result["status"]
    elapsed = ""
    if result.get("start_time"):
        elapsed = f" (started: {result['start_time'][:19]})"

    print(f"  Status: {status}{elapsed}")

    if status in ("READY", "COMPLETED", "FAILED", "STOPPED"):
        break
    time.sleep(30)

print(f"\nAlignment finished with status: {status}")
if result.get("data"):
    aligned_model_id_from_alignment = result["data"].get("model_id")
    print(f"Aligned model ID: {aligned_model_id_from_alignment}")
    print(f"Details: {json.dumps(result['data'], indent=2)}")

---

## Step 4: Deploy the Aligned Model

Once alignment is complete, we need to deploy the aligned model so we can use it for inference.

In [ ]:
# List aligned models to find ours
r = requests.get(f"{BASE_URL}/models/aligned", headers=HEADERS)
aligned_models = r.json()

print("Your aligned models:")
for m in aligned_models:
    print(f"  ID: {m['id']}  |  Status: {m.get('status', 'N/A')}")

# Use the most recently aligned model
aligned_model_id = aligned_models[-1]["id"] if aligned_models else None
print(f"\nUsing aligned model: {aligned_model_id}")

In [ ]:
# Deploy the aligned model
r = requests.post(
    f"{BASE_URL}/models/deploy-model/{aligned_model_id}",
    headers=HEADERS,
)
print(f"Deploy request: {r.status_code}")
print(r.json())

In [ ]:
# Wait for deployment
print("Waiting for model deployment...\n")

while True:
    r = requests.get(
        f"{BASE_URL}/models/deploy-model/{aligned_model_id}/status",
        headers=HEADERS,
    )
    result = r.json()
    print(f"  Deployment status: {result['status']}")
    if result["status"] in ("COMPLETED", "FAILED"):
        break
    time.sleep(15)

print(f"\nModel deployed: {aligned_model_id}")

### Quick test — ask the aligned model a question directly

Before building the full RAG pipeline, let's see how the aligned model responds on its own:

In [ ]:
def chat(model, question, context=None, max_tokens=300):
    """Send a question to a Nugen model and return the answer."""
    if context:
        prompt = (
            f"Use the following context to answer the question.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}"
        )
    else:
        prompt = question

    r = requests.post(
        f"{BASE_URL}/inference/chat/completions",
        headers=HEADERS,
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": 0.3,
        },
    )
    return r.json()["choices"][0]["message"]["content"]


# Ask the aligned model without RAG context
answer = chat(aligned_model_id, "What is the maximum weekly working hours under the Factories Act?")
print("Aligned model (no context):")
print(answer)

---

## Step 5: Build a Simple RAG Pipeline

Now let's build RAG on top of the aligned model. The idea is simple:

1. **Chunk** the document into smaller pieces
2. **Embed** each chunk using Nugen's embedding model
3. When a user asks a question, **embed the question** and **find the most similar chunks**
4. Pass those chunks as **context** to the aligned model

This gives the model precise, relevant information to work with — reducing hallucination.

### 5.1 — Chunk the document

In [ ]:
# Read the document
with open("indian_factories_act_summary.txt", "r") as f:
    full_text = f.read()


def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks by character count."""
    chunks = []
    paragraphs = text.split("\n\n")
    current = ""

    for para in paragraphs:
        if len(current) + len(para) > chunk_size and current:
            chunks.append(current.strip())
            # Keep the last `overlap` characters for continuity
            current = current[-overlap:] + "\n\n" + para
        else:
            current = current + "\n\n" + para if current else para

    if current.strip():
        chunks.append(current.strip())

    return chunks


chunks = chunk_text(full_text)
print(f"Document split into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

### 5.2 — Generate embeddings for each chunk

In [ ]:
EMBED_MODEL = "nomic-embed-text-v1.5"


def get_embeddings(texts):
    """Get embeddings for a list of texts using Nugen's embedding API."""
    r = requests.post(
        f"{BASE_URL}/inference/embeddings",
        headers=HEADERS,
        json={"model": EMBED_MODEL, "input": texts},
    )
    data = r.json()
    return [item["embedding"] for item in data["data"]]


# Embed all chunks
chunk_embeddings = get_embeddings(chunks)
chunk_matrix = np.array(chunk_embeddings)

print(f"Embedded {len(chunks)} chunks")
print(f"Embedding dimensions: {chunk_matrix.shape[1]}")

### 5.3 — Retrieval: find relevant chunks for a question

In [ ]:
def find_relevant_chunks(question, top_k=3):
    """Embed the question and find the most similar document chunks."""
    q_embedding = np.array(get_embeddings([question])[0])

    # Cosine similarity between question and each chunk
    similarities = chunk_matrix @ q_embedding / (
        np.linalg.norm(chunk_matrix, axis=1) * np.linalg.norm(q_embedding)
    )

    # Get top-k most similar chunks
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": float(similarities[idx]),
            "index": int(idx),
        })

    return results

In [ ]:
# Test retrieval
test_question = "What are the rules about child labor?"
relevant = find_relevant_chunks(test_question, top_k=2)

print(f"Question: {test_question}\n")
for r in relevant:
    print(f"Chunk {r['index']+1} (score: {r['score']:.3f}):")
    print(f"  {r['chunk'][:120]}...\n")

### 5.4 — RAG: Retrieve + Generate

In [ ]:
def ask(question, top_k=2):
    """Full RAG pipeline: retrieve relevant chunks, then generate an answer."""
    # Step 1: Retrieve
    relevant = find_relevant_chunks(question, top_k=top_k)
    context = "\n\n".join([r["chunk"] for r in relevant])

    # Step 2: Generate using the aligned model
    answer = chat(aligned_model_id, question, context=context)

    return {
        "question": question,
        "answer": answer,
        "sources": [f"Chunk {r['index']+1} (score: {r['score']:.3f})" for r in relevant],
    }

---

## Step 6: Try It Out!

Let's ask some questions about the Indian Factories Act:

In [ ]:
questions = [
    "What is the maximum number of hours a worker can work per week?",
    "What facilities must a factory provide for women workers?",
    "What are the penalties for violating the Factories Act?",
    "At what age is child labor prohibited in factories?",
    "What safety measures are required for dangerous machinery?",
]

for q in questions:
    result = ask(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {', '.join(result['sources'])}")
    print("-" * 70)

---

## Step 7: Compare — Aligned Model vs Base Model

Let's see if the aligned model performs better than the base model with the same RAG context:

In [ ]:
BASE_MODEL = "llama-v3p2-3b-reasoning"
test_q = "What welfare facilities must a factory with 300 workers provide?"

# Get the same context for both models
relevant = find_relevant_chunks(test_q, top_k=2)
context = "\n\n".join([r["chunk"] for r in relevant])

print(f"Question: {test_q}\n")
print("=" * 60)

# Base model + RAG
base_answer = chat(BASE_MODEL, test_q, context=context)
print(f"BASE MODEL ({BASE_MODEL}):")
print(base_answer)

print("\n" + "=" * 60)

# Aligned model + RAG
aligned_answer = chat(aligned_model_id, test_q, context=context)
print(f"ALIGNED MODEL ({aligned_model_id}):")
print(aligned_answer)

---

## Summary

In this tutorial, we covered the full workflow:

| Step | What we did | Nugen API used |
|------|-------------|----------------|
| 1 | Uploaded a domain document | `POST /documents` |
| 2 | Aligned a small LLM on the document | `POST /alignment-project/create` |
| 3 | Deployed the aligned model | `POST /models/deploy-model/{id}` |
| 4 | Chunked the document for retrieval | — (local Python) |
| 5 | Generated embeddings for each chunk | `POST /inference/embeddings` |
| 6 | Built a RAG pipeline: retrieve + generate | `POST /inference/chat/completions` |

**Key takeaways:**
- Alignment is a single API call — Nugen handles the training infrastructure
- Embeddings + cosine similarity give us a lightweight vector search (no database needed)
- RAG grounds the model's answers in actual document content, reducing hallucination
- The aligned model can answer domain questions more accurately than the base model

### Next steps
- Try aligning on a **larger or more complex document** (legal contracts, medical literature, technical manuals)
- Use Nugen's **benchmark API** to evaluate model quality automatically
- Add Nugen's **reranker** (`POST /inference/reranker`) to improve retrieval precision
- Scale the vector store using Qdrant or FAISS for larger document collections

---

## Cleanup

Undeploy the model when you're done to free resources:

In [ ]:
# Uncomment to undeploy the model
# r = requests.delete(
#     f"{BASE_URL}/models/undeploy-model/{aligned_model_id}",
#     headers=HEADERS,
# )
# print(f"Undeploy: {r.status_code}")